# ST-OMR Meter V5-2U — V5-2T Historical Retention V3

Single-run, read-only retention harness pinned to exact CI-green commit `55c56671fef326a96909e169ee440a22986ff71b`. It does not train, tune thresholds, or open First-30, V5 VAL, or FINAL_HOLDOUT.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import time

EXPECTED_HEAD = "55c56671fef326a96909e169ee440a22986ff71b"
EXPECTED_CI_RUN_ID = 32673186350
REPOSITORY = "khfy7wpr5p-maker/st-omr-training"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO = Path("/content/st-omr-training")
MYDRIVE = Path("/content/drive/MyDrive")

if not MYDRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
DATA_ROOT = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN"
CHECKPOINT_ROOT = MYDRIVE / "ST-OMR-METER-SPECIALISTS"
M4A_ROOT = CHECKPOINT_ROOT / "m4a-234-digit-specialist-dataset-freeze-v2"
D10_ROOT = (MYDRIVE / "ST-OMR-D10" / "stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a")
for name, path in {"DATA_ROOT": DATA_ROOT, "CHECKPOINT_ROOT": CHECKPOINT_ROOT, "M4A_ROOT": M4A_ROOT, "D10_ROOT": D10_ROOT}.items():
    if not path.is_dir():
        raise RuntimeError(f"{name} bulunamadi: {path}")
print("DRIVE CHECK = PASS")

if not REPO.exists():
    subprocess.check_call(["git", "clone", "--no-checkout", REPO_URL, str(REPO)])
elif not (REPO / ".git").is_dir():
    raise RuntimeError(f"REPO git repository degil: {REPO}")
remotes = subprocess.check_output(["git", "-C", str(REPO), "remote"], text=True).split()
if "origin" not in remotes:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "add", "origin", REPO_URL])
else:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL])
subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", EXPECTED_HEAD, "--depth", "1"])
fetched_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "FETCH_HEAD"], text=True).strip()
if fetched_head != EXPECTED_HEAD:
    raise RuntimeError(f"FETCH_HEAD mismatch: expected={EXPECTED_HEAD} actual={fetched_head}")
subprocess.check_call(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD])
actual_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if actual_head != EXPECTED_HEAD:
    raise RuntimeError(f"HEAD mismatch: {actual_head}")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Repository worktree temiz degil")
print("REPOSITORY CHECK = PASS")
print("HEAD =", actual_head)
print("CI RUN ID =", EXPECTED_CI_RUN_ID)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from st_omr_training import meter_v5_1_bbox_pilot as v51
from st_omr_training import meter_v5_2b_specialist_adaptation as v52b
from st_omr_training import meter_v5_2t_bounded_class_balanced_head_repair_v1 as v52t
from st_omr_training import meter_v5_2u_v5_2t_historical_retention_v1 as retention
print("MODULE IMPORT = PASS")

DIGIT2_FROZEN = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT2_SHA256)
DIGIT3_FROZEN = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT3_SHA256)
DIGIT4_FROZEN = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT4_SHA256)
ANN_DIR = DATA_ROOT / "annotations"
TRAINING_REPORT = ANN_DIR / v52t.TRAINING_REPORT_NAME
TRAINING_ENVELOPE = ANN_DIR / f"v5_2t_execution_envelope_{retention.V52T_IMPLEMENTATION_HEAD}.json"
CANDIDATE_DIR = ANN_DIR / v52t.CANDIDATE_DIR_NAME
DIGIT2_CANDIDATE = v52t._candidate_path(CANDIDATE_DIR, "2")
DIGIT3_CANDIDATE = v52t._candidate_path(CANDIDATE_DIR, "3")
REPORT_PATH = ANN_DIR / retention.REPORT_NAME
ENVELOPE_PATH = ANN_DIR / f"v5_2u_execution_envelope_{EXPECTED_HEAD}.json"
for name, path in {"TRAINING_REPORT": TRAINING_REPORT, "TRAINING_ENVELOPE": TRAINING_ENVELOPE, "2-AI CANDIDATE": DIGIT2_CANDIDATE, "3-AI CANDIDATE": DIGIT3_CANDIDATE}.items():
    if not path.is_file():
        raise RuntimeError(f"{name} missing: {path}")
for path in (REPORT_PATH, ENVELOPE_PATH):
    if path.exists():
        raise RuntimeError(f"Refusing overwrite/rerun: {path}")
print("EXACT INPUT BINDING = PASS")
print("OUTPUT GUARD = PASS")

required_safety = {
    "training": False,
    "autograd_grad_used": False,
    "backward": False,
    "optimizer_steps": 0,
    "checkpoint_write": False,
    "candidate_checkpoint_mutation": False,
    "runtime_threshold_tuning": False,
    "alternative_threshold_evaluated": False,
    "historical_validation_opened": True,
    "historical_retention_executed": True,
    "first30_opened": False,
    "v5_validation_opened": False,
    "final_holdout_locked": True,
    "digit4_frozen": True,
    "new_bbox": False,
    "new_crop_geometry": False,
    "new_spatial_heuristic": False,
    "production_promotion": False,
}
for key, expected in required_safety.items():
    if retention.safety_boundary().get(key) != expected:
        raise RuntimeError(f"Safety boundary mismatch: {key}")
print("READ-ONLY SAFETY BOUNDARY = PASS")
print("TRAINING=False | THRESHOLDS=FROZEN | FIRST-30=CLOSED")
print("V5_VAL=CLOSED | FINAL_HOLDOUT=LOCKED | 4-AI=FROZEN")

started = time.time()
def progress(processed, total, phase):
    if processed == 1 or processed == total or processed % 1024 == 0:
        print(phase, f"{processed}/{total}", f"| elapsed={int(time.time() - started)}s")

report = retention.run_historical_retention_v1(
    DATA_ROOT,
    m4a_root=M4A_ROOT,
    d10_root=D10_ROOT,
    digit2_frozen=DIGIT2_FROZEN,
    digit3_frozen=DIGIT3_FROZEN,
    digit4_frozen=DIGIT4_FROZEN,
    digit2_candidate=DIGIT2_CANDIDATE,
    digit3_candidate=DIGIT3_CANDIDATE,
    training_report=TRAINING_REPORT,
    execution_envelope=TRAINING_ENVELOPE,
    progress=progress,
)
if not REPORT_PATH.is_file():
    raise RuntimeError(f"Retention report missing: {REPORT_PATH}")
report_bytes = REPORT_PATH.read_bytes()
saved_report = json.loads(report_bytes.decode("utf-8"))
if saved_report != report:
    raise RuntimeError("Saved report mismatch")
if report.get("gate") not in ("PASS", "HOLD"):
    raise RuntimeError(f"Invalid retention gate: {report.get('gate')}")
for key, expected in required_safety.items():
    if report.get(key) != expected:
        raise RuntimeError(f"Saved report safety mismatch: {key}")
first30_authorized = retention.first30_authorized(report)
if first30_authorized != (report["gate"] == "PASS"):
    raise RuntimeError("First-30 authorization mismatch")
post_run_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if post_run_head != EXPECTED_HEAD:
    raise RuntimeError(f"Post-run HEAD mismatch: {post_run_head}")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Repository changed during retention")
report_sha256 = hashlib.sha256(report_bytes).hexdigest()
envelope = {
    "schema": "st-omr-meter-v5-2u-exact-sha-retention-envelope-v1",
    "repository": REPOSITORY,
    "expected_head": EXPECTED_HEAD,
    "actual_head_before_run": actual_head,
    "actual_head_after_run": post_run_head,
    "ci_run_id": EXPECTED_CI_RUN_ID,
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "retention_report_name": retention.REPORT_NAME,
    "retention_report_sha256": report_sha256,
    "gate": report["gate"],
    "reasons": report["reasons"],
    "first30_authorized": first30_authorized,
    "safety_boundary": {key: report[key] for key in required_safety},
}
v51._atomic_write_json(ENVELOPE_PATH, envelope)
envelope_sha256 = hashlib.sha256(ENVELOPE_PATH.read_bytes()).hexdigest()

print()
print("============================================")
print("V5-2U HISTORICAL RETENTION RESULT")
print("============================================")
print("GATE =", report["gate"])
print("REASONS =", report["reasons"])
for digit in ("2", "3"):
    print()
    print(f"========== {digit}-AI ==========")
    print("FROZEN =", report["frozen_metrics"][digit])
    print("CANDIDATE =", report["candidate_metrics"][digit])
    print("RETENTION =", report["per_digit_retention"][digit])
print()
print("4-AI FROZEN =", report["frozen_metrics"]["4"])
print("THRESHOLDS =", report["thresholds"])
print("THRESHOLD TUNED = False")
print("REPORT =", REPORT_PATH)
print("REPORT SHA256 =", report_sha256)
print("EXECUTION ENVELOPE =", ENVELOPE_PATH)
print("ENVELOPE SHA256 =", envelope_sha256)
if report["gate"] == "HOLD":
    print("STOP BOUNDARY = HOLD")
    print("FIRST-30 AUTHORIZED = False")
else:
    print("STOP BOUNDARY = RETENTION PASS; REVIEW BEFORE FIRST-30")
    print("FIRST-30 AUTHORIZED = True")
print("TRAINING EXECUTED = False")
print("V5 VAL = CLOSED | FINAL HOLDOUT = LOCKED")
